In [4]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
from scipy import ndimage
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy.ndimage import gaussian_filter1d

In [6]:
LABELS = {
    'NO_DOMAIN_REGION': 0,
    'DOMAIN_START': 1,
    'DOMAIN_MIDDLE': 2,
    'DOMAIN_END': 3,
}

def sliding_window_domain_smoothing(pred_arr, window_size=20, threshold=0.6):
    # Convert to binary domain/no-domain
    binary_pred = (pred_arr > 0).astype(float)
    smoothed = np.zeros_like(binary_pred)
    
    # Make window_size odd for symmetric padding
    if window_size % 2 == 0:
        window_size += 1
    
    half_window = window_size // 2
    
    # Apply sliding window
    for i in range(len(binary_pred)):
        # Define window boundaries
        start = max(0, i - half_window)
        end = min(len(binary_pred), i + half_window + 1)
        
        # Calculate fraction of domains in window
        window_values = binary_pred[start:end]
        domain_fraction = np.mean(window_values)
        
        # Keep domain prediction if fraction exceeds threshold
        smoothed[i] = 1 if domain_fraction >= threshold else 0
    
    # Convert back to original label format
    smoothed_labels = smoothed * pred_arr
    
    return smoothed_labels.astype(pred_arr.dtype)

def gaussian_domain_smoothing(pred_arr, sigma=3, threshold=0.5):
    binary_pred = (pred_arr > 0).astype(float)
    
    # Apply Gaussian smoothing along the sequence
    smoothed = gaussian_filter1d(binary_pred, sigma=sigma, mode='nearest')
    
    # Threshold to get binary domain labels
    smoothed_binary = (smoothed >= threshold).astype(float)

    return smoothed_binary.astype(pred_arr.dtype)

def find_largest_domain_blob(pred_arr):
    """Find the largest contiguous domain region in predictions."""
    binary_pred = (pred_arr > 0).astype(int)
    labeled_regions, num_regions = ndimage.label(binary_pred)
    
    if num_regions == 0:
        return np.zeros_like(pred_arr)
    
    # Find the largest region
    region_sizes = [np.sum(labeled_regions == i) for i in range(1, num_regions + 1)]
    largest_region_id = np.argmax(region_sizes) + 1
    
    # Create array with only the largest blob
    largest_blob = np.zeros_like(pred_arr)
    largest_blob_mask = (labeled_regions == largest_region_id)
    largest_blob[largest_blob_mask] = pred_arr[largest_blob_mask]
    
    return largest_blob

import numpy as np

def span_fill_domain(pred_arr):
    binary_pred = (pred_arr > 0).astype(int)
    domain_indices = np.where(binary_pred == 1)[0]
    
    if len(domain_indices) == 0:
        # No domain predicted, return empty array
        return np.zeros_like(pred_arr)
    
    start = domain_indices[0]
    end = domain_indices[-1]
    
    filled = np.zeros_like(pred_arr)
    filled[start:end + 1] = 1  # Fill the span
    
    # Optional: If you want to retain the original prediction values within the span:
    # filled[start:end + 1] = pred_arr[start:end + 1]
    
    return filled


def calculate_accuracies(true_labels, pred_arr, smoothed_pred, largest_blob):
    """Calculate various accuracy metrics."""
    # Convert to binary for accuracy calculation
    true_binary = (true_labels > 0).astype(int)
    pred_binary = (pred_arr > 0).astype(int)
    smoothed_binary = (smoothed_pred > 0).astype(int)
    largest_blob_binary = (largest_blob > 0).astype(int)
    
    # Calculate accuracies
    raw_accuracy = accuracy_score(true_binary, pred_binary)
    smoothed_accuracy = accuracy_score(true_binary, smoothed_binary)
    largest_blob_accuracy = accuracy_score(true_binary, largest_blob_binary)
    
    # Calculate precision, recall, F1 for domain class
    def get_metrics(true_bin, pred_bin):
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_bin, pred_bin, average='binary', pos_label=1, zero_division=0
        )
        return precision, recall, f1
    
    raw_prec, raw_rec, raw_f1 = get_metrics(true_binary, pred_binary)
    smooth_prec, smooth_rec, smooth_f1 = get_metrics(true_binary, smoothed_binary)
    blob_prec, blob_rec, blob_f1 = get_metrics(true_binary, largest_blob_binary)
    
    return {
        'raw': {'accuracy': raw_accuracy, 'precision': raw_prec, 'recall': raw_rec, 'f1': raw_f1},
        'smoothed': {'accuracy': smoothed_accuracy, 'precision': smooth_prec, 'recall': smooth_rec, 'f1': smooth_f1},
        'largest_blob': {'accuracy': largest_blob_accuracy, 'precision': blob_prec, 'recall': blob_rec, 'f1': blob_f1}
    }

def create_domain_plot(domain_id, true_labels, pred_arr, smoothed_pred,largest_blob,  protein_length, output_dir, metrics):
    """Create visualization comparing true vs predicted vs smoothed domain predictions."""
    
    # Setup colormap
    cmap_colors = ['white', 'blue', 'blue', 'blue']
    custom_cmap = ListedColormap(cmap_colors)
    
    # Create plot
    fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
    fig.suptitle(f"Domain {domain_id} - Prediction Comparison\n"
                f"Raw Acc: {metrics['raw']['accuracy']:.3f} | "
                f"Smoothed Acc: {metrics['smoothed']['accuracy']:.3f} | "
                f"Largest Blob Acc: {metrics['largest_blob']['accuracy']:.3f}", 
                fontsize=30)
    
    # Plot each row
    plot_data = [
        (true_labels, "Ground Truth", axes[0]),
        (pred_arr, f"Raw Prediction (F1: {metrics['raw']['f1']:.3f})", axes[1]), 
        (smoothed_pred, f"Smoothed Prediction (F1: {metrics['smoothed']['f1']:.3f})", axes[2]),
        (largest_blob, f"Largest Blob (F1: {metrics['largest_blob']['f1']:.3f})", axes[3])
    ]
    
    for data, title, ax in plot_data:
        ax.imshow(data.reshape(1, -1), cmap=custom_cmap, aspect='auto',
                 extent=[0, protein_length, 0, 1], vmin=0, vmax=3)
        ax.set_yticks([])
        ax.set_title(title, fontsize=30)
        ax.set_xlim(0, protein_length)
    
    axes[3].set_xlabel("Residue Index")
    
    # Add legend
    handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in ['white', 'blue']]
    axes[3].legend(handles, ['No Domain', 'Domain'], 
                  loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
    
    #plt.show()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{domain_id}.png"), bbox_inches='tight', dpi=150)
    plt.close(fig)

def process_predictions(prediction_path, df_path):
    """Main processing function."""
    
    # Load data
    with open(prediction_path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    subset50 = pd.read_csv(df_path)
    subset50 = subset50.set_index("domain_id")
    
    # Setup output directory
    output_dir = "output_plots_smoothed"
    os.makedirs(output_dir, exist_ok=True)
    
    # Store metrics for overall analysis
    all_metrics = []
    visualized_count = 0
    
    for entry in data:
        domain_id = entry['domain_id']
        pred_arr = np.array(entry['prediction'])
        
        if domain_id not in subset50.index:
            print(f"Skipping domain {domain_id} — not found in subset50.")
            continue
        
        # Get protein info
        row = subset50.loc[domain_id]
        protein_length = int(row['protein_length'])
        domain_start = int(row['domain_start']) - 1  # Convert to 0-based
        domain_end = int(row['domain_end'])
        
        # Create ground truth labels
        true_labels = np.full(protein_length, LABELS['NO_DOMAIN_REGION'])
        if domain_end > domain_start:
            true_labels[domain_start] = LABELS['DOMAIN_START']
            true_labels[domain_start + 1:domain_end - 1] = LABELS['DOMAIN_MIDDLE']
            true_labels[domain_end - 1] = LABELS['DOMAIN_END']
        
        # Resize prediction array to match protein length
        if len(pred_arr) != protein_length:
            resized_pred = np.full(protein_length, LABELS['NO_DOMAIN_REGION'])
            copy_length = min(len(pred_arr), protein_length)
            resized_pred[:copy_length] = pred_arr[:copy_length]
            pred_arr = resized_pred
        
        # Calculate smoothed predictions and largest blob
        smoothed_pred = gaussian_domain_smoothing(pred_arr, sigma=15, threshold=0.3)
        largest_blob = find_largest_domain_blob(smoothed_pred)
        
        # Calculate metrics
        metrics = calculate_accuracies(true_labels, pred_arr, smoothed_pred, largest_blob)
        all_metrics.append(metrics)
        
        # Create visualization
        if visualized_count % 10 == 0:
            create_domain_plot(domain_id, true_labels, pred_arr, smoothed_pred, largest_blob, protein_length, output_dir, metrics)
        
        visualized_count += 1
        """print(f"Generated visualization {visualized_count} for domain {domain_id}")
        print(f"  Raw: Acc={metrics['raw']['accuracy']:.3f}, F1={metrics['raw']['f1']:.3f}")
        print(f"  Smoothed: Acc={metrics['smoothed']['accuracy']:.3f}, F1={metrics['smoothed']['f1']:.3f}")
        print(f"  Largest Blob: Acc={metrics['largest_blob']['accuracy']:.3f}, F1={metrics['largest_blob']['f1']:.3f}")"""
    
    # Calculate overall statistics
    if all_metrics:
        avg_metrics = {}
        for method in ['raw', 'smoothed', 'largest_blob']:
            avg_metrics[method] = {
                'accuracy': np.mean([m[method]['accuracy'] for m in all_metrics]),
                'precision': np.mean([m[method]['precision'] for m in all_metrics]),
                'recall': np.mean([m[method]['recall'] for m in all_metrics]),
                'f1': np.mean([m[method]['f1'] for m in all_metrics])
            }
        
        print(f"\n=== OVERALL PERFORMANCE SUMMARY ===")
        print(f"{'Method':<15} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10}")
        print("-" * 55)
        for method, metrics_dict in avg_metrics.items():
            print(f"{method.replace('_', ' ').title():<15} "
                  f"{metrics_dict['accuracy']:<10.3f} "
                  f"{metrics_dict['precision']:<10.3f} "
                  f"{metrics_dict['recall']:<10.3f} "
                  f"{metrics_dict['f1']:<10.3f}")
    
    print(f"\nCompleted! Generated {visualized_count} visualizations in '{output_dir}' directory.")

process_predictions("/Users/bene/Developer/protein-prediction-project/data/test-predictions-boundary.json", "/Users/bene/Developer/protein-prediction-project/datasets/v5/test_split.csv")


=== OVERALL PERFORMANCE SUMMARY ===
Method          Accuracy   Precision  Recall     F1-Score  
-------------------------------------------------------
Raw             0.769      0.750      0.819      0.758     
Smoothed        0.766      0.729      0.898      0.776     
Largest Blob    0.770      0.733      0.871      0.768     

Completed! Generated 1399 visualizations in 'output_plots_smoothed' directory.


In [16]:
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d

def extract_domain_boundaries(sigma=15, threshold=0.3):
    with open("/Users/bene/Developer/protein-prediction-project/data/test-predictions-boundary.json", 'r') as f:
        data = [json.loads(line) for line in f]
    
    results = []
    
    for entry in data:
        domain_id = entry['domain_id']
        pred_arr = np.array(entry['prediction'])
        
        # Apply Gaussian smoothing
        binary_pred = (pred_arr > 0).astype(float)
        smoothed = gaussian_filter1d(binary_pred, sigma=sigma, mode='nearest')
        smoothed_binary = (smoothed >= threshold).astype(int)
        
        # Find domain boundaries
        domain_indices = np.where(smoothed_binary == 1)[0]
        
        if len(domain_indices) > 0:
            start_idx = int(domain_indices[0]) + 1
            end_idx = int(domain_indices[-1]) + 1
        else:
            # No domain found
            start_idx = 0
            end_idx = 0
        
        results.append({
            'domain_id': domain_id,
            'start_idx': start_idx,
            'end_idx': end_idx
        })
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    return df

predicted_df = extract_domain_boundaries()

In [17]:
import pandas as pd

df = pd.read_csv("/Users/bene/Developer/protein-prediction-project/datasets/v5/test_split.csv")
df.to_csv("../../datasets/predicted_boundaries_v1/test_split.csv", index=False)

In [18]:
merged = pd.merge(predicted_df, df, on='domain_id', how='inner')
merged.rename(columns={'domain_start': 'real_domain_start', 'domain_end': 'real_domain_end'}, inplace=True)
merged.rename(columns={'start_idx': 'domain_start', 'end_idx': 'domain_end'}, inplace=True)
merged.to_csv("../../datasets/predicted_boundaries_v1/test_split.csv", index=False)